In [ ]:
from qiskit.circuit import Instruction
from numpy import array
import numpy as np
import qiskit.circuit.library as qGate
from qiskit.circuit.library import UnitaryGate

from qatg import QATG
from qatg import QATGFault

In [ ]:
flist = np.load('faults.npy', allow_pickle=True)
class mySXFault(QATGFault):
	def __init__(self, fgate):
		super(mySXFault, self).__init__(qGate.SXGate, 0, "gateType: SX, qubits: 0")
		self.fgate = UnitaryGate(fgate)
	def createOriginalGate(self):
		return qGate.SXGate()
	def createFaultyGate(self, faultfreeGate):
		return self.fgate

In [ ]:
nfaults = len(flist)
nfaults

In [ ]:
generator = QATG(circuitSize = 1, basisSingleQubitGateSet = [qGate.SXGate, qGate.XGate, qGate.RZGate], circuitInitializedStates = {1: [1, 0]}, minRequiredStateFidelity = 0.1, verbose = True,  maxTestTemplateSize=30)
#generator = QATG(circuitSize = 1, basisGateSet = [qGate.UGate], circuitInitializedStates = {1: [1, 0]}, minRequiredEffectSize = 3, verbose = True, maxTestTemplateSize=3)
#configurationList = generator.createTestConfiguration([RotationFault(qGate.XGate, 0, 0.1, 0, 0)])
#Flist = [[mySXFault(flist[0]), mySXFault(flist[1]), mySXFault(flist[2]), mySXFault(flist[3])]]
Flist_4 = [[mySXFault(flist[i]), mySXFault(flist[i+1]), mySXFault(flist[-i-1]), mySXFault(flist[-i-2])] for i in range(nfaults//2)]
Flist_2 = [[mySXFault(flist[i]), mySXFault(flist[i+1])] for i in range(nfaults - 1)]
Flist = Flist_2 + Flist_4
configurationList = generator.createTestConfigurationCompressed(Flist)

In [ ]:
class Faulty_QC:
    def __init__(self, fault_model):
        self.faultObject = fault_model
    def generate_faulty_qc(self, qc_t):
        qc_f = qc_t.copy_empty_like()
        qbIndexes = self.faultObject.getQubits()
        for ci in qc_t:
          gates = ci.operation
          qubits = [qc_t.find_bit(qb).index for qb in ci.qubits]
          if self.faultObject.isSameGateType(gates) and qubits == qbIndexes:
            qc_f.append(self.faultObject.createFaultyGate(gates), qbIndexes)
          else:
            qc_f.append(ci)
        return qc_f
    def run(self, qc, shots=1024):
        if (type(qc) != QuantumCircuit):
            raise(ValueError("qc should be a QuantumCircuit"))
        qc_t = transpile(qc, basis_gates = ['sx', 'rz', 'cx'])
        qc_f = self.generate_faulty_qc(qc_t)
        return AerSimulator().run(qc_f, shots=shots).result().get_counts()

In [ ]:
from qiskit.circuit import QuantumCircuit
from qiskit import transpile
from qiskit_aer import AerSimulator

print("Result fault free")
fsim = AerSimulator()
for i in range(len(configurationList)):
    qc = configurationList[i].faultfreeQCKT
    print('test fail rate for test circuit', i, ":", (100000-fsim.run(qc, shots = 100000).result().get_counts()['0'])/100000)

for f in flist:
    print("\nresult for fault", f)
    fsim = Faulty_QC(mySXFault(f))
    for i in range(len(configurationList)):
        qc = configurationList[i].faultfreeQCKT
        print('test fail rate for test circuit', i, ":", fsim.run(qc, 100000)['1']/100000)

In [ ]:
from qiskit.qasm2 import dump

for i in range(len(configurationList)):
    dump(configurationList[i].faultfreeQCKT, 'test_circuit_{}.qasm'.format(i))

In [ ]:
len(configurationList)